# Taxi Trip Data Processing
Monthly chunk pipeline for NYC TLC yellow taxi data.

In [26]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa

In [27]:
# --- Configuration ---
YEAR = 2022
TAXI_TYPE = 'yellow'
INPUT_DIR = Path('../raw')
OUTPUT_DIR = Path('../processed')
REPORTS_DIR = Path('../reports')

# --- Add constants for data cleaning ---
FILTER_SPEED_ABOVE = 100
TRIP_LENGTH_FILTER_MIN = 1  # in minutes
TRIP_LENGTH_FILTER_MAX = 240  # in minutes

In [ ]:
# --- Create required directories ---
OUTPUT_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)
temp_dir = OUTPUT_DIR / 'temp_cleaned'
temp_dir.mkdir(exist_ok=True)

In [28]:
def clean_data(df: pd.DataFrame, initial_rows: int) -> (pd.DataFrame, pd.DataFrame):
    """
    Applies a series of data cleaning and quality assurance steps to a chunk of taxi trip data.

    Args:
        df: The raw taxi trip DataFrame chunk.
        initial_rows: The number of rows in the original, unfiltered DataFrame for percentage calculation.

    Returns:
        A tuple containing the cleaned DataFrame and a summary of the QA checks for the chunk.
    """
    qa_summary = {}

    # Convert to datetime and calculate trip duration
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    df['trip_duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

    # Define and apply filters
    cost_cols = ['fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount']
    filters = {
        'trip_outside_year': (df['tpep_pickup_datetime'].dt.year != YEAR) | (df['tpep_dropoff_datetime'].dt.year != YEAR),
        'invalid_trip_duration': df['tpep_dropoff_datetime'] <= df['tpep_pickup_datetime'],
        'zero_or_negative_trip_distance': df['trip_distance'] <= 0,
        'zero_passenger_count': df['passenger_count'] == 0,
        'negative_cost_values': (df[cost_cols] < 0).any(axis=1),
        'outlier_trip_duration': (df['trip_duration_minutes'] < TRIP_LENGTH_FILTER_MIN) | (df['trip_duration_minutes'] > TRIP_LENGTH_FILTER_MAX)
    }

    for name, mask in filters.items():
        qa_summary[name] = mask.sum()
        df = df[~mask]

    # Calculate trip speed and apply speed filter
    non_zero_duration_mask = df['trip_duration_minutes'] > 0
    df['trip_speed_mph'] = 0.0
    df.loc[non_zero_duration_mask, 'trip_speed_mph'] = df.loc[non_zero_duration_mask, 'trip_distance'] / (df.loc[non_zero_duration_mask, 'trip_duration_minutes'] / 60)
    
    speed_filter = df['trip_speed_mph'] > FILTER_SPEED_ABOVE
    qa_summary['outlier_trip_speed'] = speed_filter.sum()
    df = df[~speed_filter]

    summary_df = pd.DataFrame.from_dict(qa_summary, orient='index', columns=['records_removed'])
    return df, summary_df

In [ ]:
def process_chunk(file_path):
    """Processes a single chunk of data: cleans, saves temp file, and returns data for aggregation."""
    print(f"Processing chunk: {file_path.name}...")
    df_chunk = pd.read_parquet(file_path)
    initial_rows = len(df_chunk)
    cleaned_df, qa_summary = clean_data(df_chunk, initial_rows)
    
    temp_output_path = temp_dir / f'cleaned_{file_path.name}'
    cleaned_df.to_parquet(temp_output_path, index=False)
    
    kpi_cols = ['tpep_pickup_datetime', 'trip_duration_minutes', 'trip_speed_mph']
    return initial_rows, qa_summary, cleaned_df[kpi_cols], temp_output_path

In [ ]:
def process_all_trip_files(trip_files):
    """Processes all monthly chunk files and returns aggregated intermediaries."""
    if not trip_files:
        print(f"Error: No Parquet files found in '{INPUT_DIR}' for {YEAR}.\nPlease run the download script first.")
        return None

    results = [process_chunk(fp) for fp in trip_files]
    total_initial_rows = sum(r[0] for r in results)
    qa_summaries = [r[1] for r in results]
    kpi_frames = [r[2] for r in results]
    cleaned_paths = [r[3] for r in results]
    return total_initial_rows, qa_summaries, kpi_frames, cleaned_paths

In [ ]:
def aggregate_qa_reports(all_qa_summaries, total_initial_rows, reports_dir, year):
    if not all_qa_summaries:
        print("No QA summaries to aggregate.")
        return None
    print("Aggregating QA reports...")
    final_qa_summary = pd.concat(all_qa_summaries).groupby(level=0).sum()
    if total_initial_rows > 0:
        final_qa_summary['percentage_of_total'] = (final_qa_summary['records_removed'] / total_initial_rows) * 100
    else:
        final_qa_summary['percentage_of_total'] = 0.0
    qa_report_path = reports_dir / f'qa_summary_{year}.csv'
    final_qa_summary.to_csv(qa_report_path)
    print(f"QA summary saved to {qa_report_path}")
    return final_qa_summary, qa_report_path

In [ ]:
def build_kpi_source(kpi_frames):
    if not kpi_frames:
        raise ValueError("No KPI frames were generated from processed chunks.")
    print("Combining data for KPI calculations...")
    return pd.concat(kpi_frames, ignore_index=True)

In [29]:
def feature_engineering_and_kpis(df: pd.DataFrame, year: int):
    """
    Engineers time-based features and calculates daily, weekly, and monthly KPIs.
    Produces KPI CSVs:
      - processed/kpi_daily_YYYY.csv
      - processed/kpi_weekly_YYYY.csv
      - processed/kpi_monthly_YYYY.csv
    Also writes KPI definitions to processed/kpi_definitions_YYYY.csv.
    """
    print("Starting feature engineering and KPI calculation on combined data...")

    # Ensure datetime and basic features
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['pickup_date'] = df['tpep_pickup_datetime'].dt.date
    df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
    df['day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek    # 0=Monday
    df['day_name'] = df['tpep_pickup_datetime'].dt.day_name()
    df['week_of_year'] = df['tpep_pickup_datetime'].dt.isocalendar().week
    df['month'] = df['tpep_pickup_datetime'].dt.month
    df['month_name'] = df['tpep_pickup_datetime'].dt.month_name()

    # Annual totals for percent calculations
    annual_total_trips = len(df)
    mean_daily_trips = df.groupby('pickup_date')['tpep_pickup_datetime'].count().mean() if annual_total_trips>0 else 0.0

    # KPI aggregations (common aggregations used across periods)
    agg_funcs = {
        'total_trips': ('tpep_pickup_datetime', 'count'),
        'p50_trip_duration': ('trip_duration_minutes', lambda x: x.quantile(0.5)),
        'p95_trip_duration': ('trip_duration_minutes', lambda x: x.quantile(0.95)),
        'p50_trip_speed': ('trip_speed_mph', lambda x: x.quantile(0.5))
    }

    # Daily KPIs
    daily = df.groupby('pickup_date').agg(**agg_funcs).reset_index()
    # Daily index (baseline 100 = average daily trips)
    if mean_daily_trips > 0:
        daily['index_100'] = (daily['total_trips'] / mean_daily_trips) * 100
    else:
        daily['index_100'] = 0.0
    daily_path = OUTPUT_DIR / f'kpi_daily_{year}.csv'
    daily.to_csv(daily_path, index=False)
    print(f"Daily KPIs saved to {daily_path}")

    # Weekly KPIs (by ISO week number)
    weekly = df.groupby('week_of_year').agg(**agg_funcs).reset_index().rename(columns={'week_of_year':'week'})
    weekly_path = OUTPUT_DIR / f'kpi_weekly_{year}.csv'
    weekly.to_csv(weekly_path, index=False)
    print(f"Weekly KPIs saved to {weekly_path}")

    # Monthly KPIs + percent of annual total
    monthly = df.groupby('month').agg(**agg_funcs).reset_index()
    monthly['month_name'] = monthly['month'].apply(lambda m: pd.to_datetime(str(m), format='%m').month_name())
    if annual_total_trips > 0:
        # compute percent of annual trips per month
        month_totals = df.groupby('month')['tpep_pickup_datetime'].count()
        monthly = monthly.merge(month_totals.rename('month_total_trips'), on='month')
        monthly['pct_of_annual'] = (monthly['month_total_trips'] / annual_total_trips) * 100
    else:
        monthly['month_total_trips'] = 0
        monthly['pct_of_annual'] = 0.0
    monthly_path = OUTPUT_DIR / f'kpi_monthly_{year}.csv'
    monthly.to_csv(monthly_path, index=False)
    print(f"Monthly KPIs saved to {monthly_path}")

    # Additional KPIs: p95 by weekday and p50 speed by hour (auxiliary tables)
    p95_by_weekday = df.groupby('day_of_week').agg(
        p95_trip_duration=('trip_duration_minutes', lambda x: x.quantile(0.95)),
        mean_trips=('tpep_pickup_datetime','count')
    ).reset_index()
    
    # IndexDow: normalize weekday mean trips to overall mean daily trips
    if mean_daily_trips > 0:
        p95_by_weekday['indexdow_100'] = (p95_by_weekday['mean_trips'] / mean_daily_trips) * 100
    else:
        p95_by_weekday['indexdow_100'] = 0.0

    p50_speed_by_hour = df.groupby('pickup_hour').agg(
        p50_trip_speed=('trip_speed_mph', lambda x: x.quantile(0.5)),
        mean_trips=('tpep_pickup_datetime','count')
    ).reset_index()

    # Save auxiliary KPI tables into the processed folder for inspection (optional)
    aux_weekday_path = OUTPUT_DIR / f'kpi_weekday_{year}.csv'
    p95_by_weekday.to_csv(aux_weekday_path, index=False)
    aux_hour_path = OUTPUT_DIR / f'kpi_hourly_speed_{year}.csv'
    p50_speed_by_hour.to_csv(aux_hour_path, index=False)

    # KPI definitions: name → target → formula → unit
    kpi_definitions = [
        ('total_trips', 'Increase riders/trips', 'Count of trips in period', 'trips'),
        ('p50_trip_duration', 'Median trip duration', '50th percentile of trip_duration_minutes', 'minutes'),
        ('p95_trip_duration', 'Tail latency of trips', '95th percentile of trip_duration_minutes', 'minutes'),
        ('p50_trip_speed', 'Typical trip speed', '50th percentile of trip_speed_mph', 'mph'),
        ('pct_of_annual', 'Share of annual volume', 'month_total_trips / annual_total_trips * 100', 'percent'),
        ('index_100', 'Daily volume index (baseline=100)', 'total_trips / mean_daily_trips * 100', 'index(100)'),
        ('indexdow_100', 'Weekday index (baseline=100)', 'mean_trips_by_weekday / mean_daily_trips * 100', 'index(100)')
    ]
    kpi_defs_df = pd.DataFrame(kpi_definitions, columns=['kpi_name','target','formula','unit'])
    kpi_defs_path = OUTPUT_DIR / f'kpi_definitions_{year}.csv'
    kpi_defs_df.to_csv(kpi_defs_path, index=False)
    print(f"KPI definitions saved to {kpi_defs_path}")

    # Print short summary
    print(f"Annual trips: {annual_total_trips}, mean daily trips: {mean_daily_trips:.1f}")
    print("KPI generation complete.")

In [ ]:
def write_final_cleaned_dataset(cleaned_chunk_paths, output_dir, taxi_type, year):
    if not cleaned_chunk_paths:
        raise ValueError("No cleaned chunk paths provided for final dataset.")
    print("Combining cleaned data into final Parquet file...")
    final_cleaned_path = output_dir / f'cleaned_{taxi_type}_tripdata_{year}.parquet'
    combine_parquet_files(cleaned_chunk_paths, final_cleaned_path)
    print(f"Final cleaned data saved to {final_cleaned_path}")
    return final_cleaned_path

In [ ]:
def cleanup_temp_directory(temp_dir):
    if temp_dir.exists():
        print("Cleaning up temporary files...")
        for temp_file in temp_dir.iterdir():
            temp_file.unlink()
        temp_dir.rmdir()

In [ ]:
def sample_cleaned_trips(path, sample_size=10000):
    df = pd.read_parquet(path)
    return df.sample(min(sample_size, len(df)))

In [ ]:
# --- Pipeline Execution ---
trip_files = sorted(INPUT_DIR.glob(f"{TAXI_TYPE}_tripdata_{YEAR}-*.parquet"))
processing_output = process_all_trip_files(trip_files)

if processing_output:
    total_initial_rows, qa_summaries, kpi_frames, cleaned_paths = processing_output
    qa_result = aggregate_qa_reports(qa_summaries, total_initial_rows, REPORTS_DIR, YEAR)
    if qa_result:
        final_qa_summary, qa_report_path = qa_result
    kpi_source_df = build_kpi_source(kpi_frames)
    feature_engineering_and_kpis(kpi_source_df, YEAR)
    final_cleaned_path = write_final_cleaned_dataset(cleaned_paths, OUTPUT_DIR, TAXI_TYPE, YEAR)
    df = sample_cleaned_trips(final_cleaned_path)
    cleanup_temp_directory(temp_dir)